## Sentiment Labeling
- Use a pretrained NLP model (e.g., HuggingFace’s transformers with distilbert-base-uncased-finetuned-sst-2-english) or scikit‑learn with TF‑IDF + Logistic Regression.

- Load test.csv.

- Preprocess text (lowercase, remove stopwords, punctuation).

- Apply sentiment classifier → assign labels: Positive, Negative, Neutral.

- Save augmented dataset with a new column Sentiment.

In [9]:
from transformers import pipeline
import pandas as pd

df = pd.read_csv("test.csv")
sentiment_model = pipeline("sentiment-analysis")

df["Sentiment"] = df["Message"].apply(lambda x: sentiment_model(x)[0]['label'])

ImportError: cannot import name 'pipeline' from 'transformers' (c:\Users\Muhammad Haris\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\__init__.py)

## Exploratory Data Analysis (EDA)

- Check dataset structure: missing values, message length distribution.

- Visualizations:

    - Sentiment distribution (bar chart).

    - Sentiment trends over time (line plot).

    - Word clouds for positive vs negative messages

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(x="Sentiment", data=df)
plt.show()

## Employee Score Calculation
- Define scoring: Positive = +1, Negative = -1, Neutral = 0.

- Group by EmployeeID and Month.

- Aggregate scores.

In [ ]:
df["Score"] = df["Sentiment"].map({"POSITIVE":1, "NEGATIVE":-1, "NEUTRAL":0})
monthly_scores = df.groupby(["EmployeeID","Month"])["Score"].sum().reset_index()

## Employee Ranking
- Sort employees by monthly score.

- Extract Top 3 Positive and Top 3 Negative employees.

In [ ]:
top_positive = monthly_scores.groupby("Month").apply(lambda x: x.nlargest(3,"Score"))
top_negative = monthly_scores.groupby("Month").apply(lambda x: x.nsmallest(3,"Score"))

## Flight Risk Identification
-   Flag employees with ≥4 negative messages in any rolling 30‑day window.

-   Use a rolling count per employee.

In [ ]:
df["NegativeFlag"] = (df["Sentiment"]=="NEGATIVE").astype(int)
flight_risk = df.groupby(["EmployeeID","Month"])["NegativeFlag"].sum()
risk_employees = flight_risk[flight_risk >= 4].index

## Predictive Modeling
-   Features: message frequency, avg message length, word count, sentiment ratio.

-   Target: monthly sentiment score.

-   Train/test split → Linear Regression.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X = monthly_scores[["MessageCount","AvgLength","WordCount"]]
y = monthly_scores["Score"]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
model = LinearRegression().fit(X_train,y_train)